# 🎬 Question Answering over Graph Databases with LangChain & Groq

**Masterclass: End-to-End GraphCypherQAChain, Multi-Dataset Ingestion & Conversational Graph Search**

In modern enterprise applications, vast amounts of critical business knowledge exist as interconnected networks: supply chains, customer social graphs, corporate ownership hierarchies, and media catalogs. While standard relational databases require complex multi-table SQL joins and vector databases only perform fuzzy semantic similarity, **Graph Databases (like Neo4j)** represent relationships as native first-class citizens.

By pairing **LangChain's `GraphCypherQAChain`** with ultra-fast LLMs (**Groq `openai/gpt-oss-120b`**), users can ask natural questions in plain English. The LLM translates the question into an optimal **Cypher query**, runs it directly against **Neo4j Aura Cloud**, and synthesizes a concise, grounded natural language answer with zero hallucinations.

---

![Graph Cypher QA Workflow](workflow_graph_cypher_qa.png)

<details>
<summary><b>🔍 Click to Expand Interactive Mermaid Architecture Diagram</b></summary>

```mermaid
flowchart TD
    subgraph UserInterface["User Query Plane"]
        Q(["Natural Language Question<br/>e.g. 'Who directed Casino?'"])
    end

    subgraph LangChain_Engine["LangChain GraphCypherQAChain Core"]
        SCHEMA["Neo4j Schema Provider<br/>(Labels, Properties, Relationships)"]
        CYPHER_PROMPT["Cypher Generation Prompt<br/>(Schema + Instructions)"]
        GROQ_LLM["Groq High-Speed LLM<br/><b>openai/gpt-oss-120b</b>"]
        CYPHER_EXTRACT["Generated Cypher Query<br/><code>MATCH (p)-[:DIRECTED]->(m)</code>"]
    end

    subgraph DatabaseLayer["Neo4j Aura Cloud Database"]
        NEO4J[("Neo4j Aura Cloud<br/>Property Graph Store")]
        EXEC["Cypher Query Executor<br/>(Graph Index Traversal)"]
        RECORDS["Raw Database Records<br/>(Entities, Tuples, Aggregates)"]
    end

    subgraph Synthesis["Answer Synthesis Plane"]
        SYNTH_PROMPT["Final Answer Prompt<br/>(Question + Graph Records)"]
        FINAL_LLM["Groq Reasoning Engine<br/><b>openai/gpt-oss-120b</b>"]
        ANSWER(["Grounded Conversational Response"])
    end

    Q --> CYPHER_PROMPT
    SCHEMA --> CYPHER_PROMPT
    CYPHER_PROMPT --> GROQ_LLM
    GROQ_LLM --> CYPHER_EXTRACT
    CYPHER_EXTRACT --> EXEC
    EXEC --> NEO4J
    NEO4J --> RECORDS
    RECORDS --> SYNTH_PROMPT
    Q --> SYNTH_PROMPT
    SYNTH_PROMPT --> FINAL_LLM
    FINAL_LLM --> ANSWER
```
</details>

---


In [12]:
"""
Here we set up our workspace by loading our secret keys from our .env configuration file.
We import the official LangChain Neo4j integration package, our Groq language model,
and Pandas for creating clean, easy-to-read comparison scorecards.
"""
import os
import sys
import json
import time
from dotenv import load_dotenv

# Load secret credentials from .env
load_dotenv()

# Verify that essential keys are present
assert os.getenv("GROQ_API_KEY"), "Missing GROQ_API_KEY in .env"
assert os.getenv("NEO4J_URI"), "Missing NEO4J_URI in .env"
assert os.getenv("NEO4J_PASSWORD"), "Missing NEO4J_PASSWORD in .env"

import pandas as pd
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_groq import ChatGroq

print("✅ Environment successfully configured and modern LangChain Neo4j packages imported!")


✅ Environment successfully configured and modern LangChain Neo4j packages imported!


In [13]:
"""
Now we connect directly to our live Neo4j Aura Cloud instance.
We supply our secure connection URL, username, password, and target database.
Once connected, we ask the database to introduce itself and verify that our connection is healthy.
"""
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_user = os.getenv("NEO4J_USERNAME", "neo4j")
neo4j_pwd = os.getenv("NEO4J_PASSWORD")
neo4j_db = os.getenv("NEO4J_DATABASE", "neo4j")

# Connect to the cloud graph database
graph = Neo4jGraph(
    url=neo4j_uri,
    username=neo4j_user,
    password=neo4j_pwd,
    database=neo4j_db
)

print(f"✅ Successfully connected to Neo4j Aura Cloud at: {neo4j_uri}")


✅ Successfully connected to Neo4j Aura Cloud at: neo4j+ssc://82bce487.databases.neo4j.io


In [14]:
"""
To make sure our notebook runs reliably from scratch without conflicting with previous runs,
we clear out any existing nodes and relationships from the database.
This gives us a clean slate ready to receive our two new datasets.
"""
# Clear the active database
graph.query("MATCH (n) DETACH DELETE n")

# Verify node count is zero
count_check = graph.query("MATCH (n) RETURN count(n) AS total_nodes")
print(f"🧹 Database reset complete! Current active nodes: {count_check[0]['total_nodes']}")


🧹 Database reset complete! Current active nodes: 0


In [15]:
"""
Here we ingest our first real-world dataset: The Movies Knowledge Graph.
We read data from a public CSV file containing movies, their release dates, IMDb ratings,
directors, actors, and genres.
Using Cypher's MERGE command, we create Movie, Person, and Genre nodes,
and link them together with DIRECTED, ACTED_IN, and IN_GENRE relationships.
"""
"""Sample data -
movieId,released,title,actors,director,genres,imdbRating
1,1995-11-22,Toy Story,Jim Varney|Tim Allen|Tom Hanks|Don Rickles,John Lasseter,Adventure|Animation|Children|Comedy|Fantasy,8.3
"""

movies_query = """
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' AS row

MERGE (m:Movie {id: row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)

FOREACH (director IN split(row.director, '|') |
    MERGE (p:Person {name: trim(director)})
    MERGE (p)-[:DIRECTED]->(m))

FOREACH (actor IN split(row.actors, '|') | 
    MERGE (p:Person {name: trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))

FOREACH (genre IN split(row.genres, '|') | 
    MERGE (g:Genre {name: trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))
"""

print("⏳ Ingesting Movies, Actors, Directors, and Genres into Neo4j...")
graph.query(movies_query)

# Verify movie counts
movie_count = graph.query("MATCH (m:Movie) RETURN count(m) AS total_movies")[0]["total_movies"]
actor_count = graph.query("MATCH (p:Person)-[:ACTED_IN]->() RETURN count(DISTINCT p) AS total_actors")[0]["total_actors"]
print(f"✅ Movies Knowledge Graph Ingested: {movie_count} Movies | {actor_count} Unique Actors!")


⏳ Ingesting Movies, Actors, Directors, and Genres into Neo4j...
✅ Movies Knowledge Graph Ingested: 299 Movies | 967 Unique Actors!


In [16]:
"""
Now we ingest our second dataset: A rich Social Network Graph.
We load Users, their Posts, and the connections between them (FRIEND and LIKES relationships).
This gives our database both entertainment media knowledge and social interaction knowledge!
"""

""" - sample data - 
userId,name,age,city
1,John,30,New York
2,Sarah,25,San Francisco

postId,content,userId,timestamp
1,Hello World!,1,2024-07-14T12:34:56
2,My first post!,2,2024-07-14T13:00:00

userId1,userId2,type
1,2,FRIEND
1,3,FRIEND
"""
# 1. Load Users
load_users_query = """
LOAD CSV WITH HEADERS FROM 'https://raw.githubusercontent.com/krishnaik06/graph-dataset/main/users_social.csv' AS row
CREATE (:User {userId: toInteger(row.userId), name: row.name, age: toInteger(row.age), city: row.city})
"""
graph.query(load_users_query)

# 2. Load Posts and link with POSTED relationship
load_posts_query = """
LOAD CSV WITH HEADERS FROM 'https://raw.githubusercontent.com/krishnaik06/graph-dataset/main/posts.csv' AS row 
MATCH (u:User {userId: toInteger(row.userId)})
CREATE (u)-[:POSTED]->(:Post {postId: toInteger(row.postId), content: row.content, timestamp: datetime(row.timestamp)})
"""
graph.query(load_posts_query)

# 3. Create FRIEND and LIKES Relationships between Users
load_relationships_query = """
LOAD CSV WITH HEADERS FROM 'https://raw.githubusercontent.com/krishnaik06/graph-dataset/main/relationships.csv' AS row
MATCH (u1:User {userId: toInteger(row.userId1)}), (u2:User {userId: toInteger(row.userId2)})
CREATE (u1)-[:FRIEND]->(u2)
CREATE (u1)-[:LIKES]->(u2)
"""
graph.query(load_relationships_query)

user_count = graph.query("MATCH (u:User) RETURN count(u) AS c")[0]["c"]
post_count = graph.query("MATCH (p:Post) RETURN count(p) AS c")[0]["c"]
print(f"✅ Social Network Graph Ingested: {user_count} Users | {post_count} Posts!")


✅ Social Network Graph Ingested: 5 Users | 5 Posts!


In [17]:
"""
Now we refresh our graph schema so that LangChain knows the exact names of all
labels, relationship types, and properties inside our database.
We display a clear summary so we can see the full structure of our graph assets.
"""
graph.refresh_schema()

print("=== 📋 DYNAMIC GRAPH SCHEMA ===")
print(graph.schema)

# Tabular node breakdown
node_breakdown = graph.query("""
MATCH (n)
RETURN labels(n)[0] AS Label, count(n) AS TotalCount
ORDER BY TotalCount DESC
""")

print("\n=== 📊 NODE DISTRIBUTION BY LABEL ===")
display(pd.DataFrame(node_breakdown))


=== 📋 DYNAMIC GRAPH SCHEMA ===
Node properties:
Person {name: STRING}
Movie {id: STRING, released: DATE, title: STRING, imdbRating: FLOAT}
Genre {name: STRING}
User {name: STRING, city: STRING, userId: INTEGER, age: INTEGER}
Post {content: STRING, postId: INTEGER, timestamp: DATE_TIME}
Relationship properties:

The relationships:
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)
(:User)-[:POSTED]->(:Post)
(:User)-[:LIKES]->(:User)
(:User)-[:FRIEND]->(:User)

=== 📊 NODE DISTRIBUTION BY LABEL ===


,Label,TotalCount
0,Person,1239
1,Movie,299
2,Genre,19
3,User,5
4,Post,5


In [18]:
"""
We initialize our Groq language model using 'openai/gpt-oss-120b'.
We set the temperature to 0.0 to ensure strict factual consistency and high-precision
Cypher query writing, preventing creative liberties that might produce invalid database queries.
"""
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.0
)

print(f"🤖 Groq LLM ready: {llm.model_name}")


🤖 Groq LLM ready: openai/gpt-oss-120b


In [19]:
"""
Here we construct our GraphCypherQAChain.
This chain acts as the brain:
1. It feeds the graph schema and user question to the LLM.
2. The LLM translates the question into a clean Cypher query.
3. The chain executes that query against Neo4j.
4. The chain feeds the database records back to the LLM to write a friendly natural language response!
We set return_intermediate_steps=True so we can inspect every step of the process.
"""
qa_chain = GraphCypherQAChain.from_llm(
    graph=graph,
    llm=llm,
    verbose=True,
    return_intermediate_steps=True,
    allow_dangerous_requests=True
)

print("🚀 GraphCypherQAChain assembled and ready for natural language question answering!")


🚀 GraphCypherQAChain assembled and ready for natural language question answering!


In [20]:
"""
Let us ask our first question: finding who directed the movie Casino.
We run the question through our chain and display the generated Cypher query,
the raw records returned by Neo4j, and the final synthesized answer.
"""
q1 = "Who was the director of the movie Casino?"
res1 = qa_chain.invoke({"query": q1})

print(f"❓ Question: {q1}")
print(f"💻 Generated Cypher Query: {res1['intermediate_steps'][0]['query']}")
print(f"📦 Graph Records Returned: {res1['intermediate_steps'][1]['context']}")
print(f"💡 Final Answer: {res1['result']}")




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie {title: "Casino"})
RETURN p.name AS director_name
Full Context:
[{'director_name': 'Martin Scorsese'}]

> Finished chain.
❓ Question: Who was the director of the movie Casino?
💻 Generated Cypher Query: MATCH (p:Person)-[:DIRECTED]->(m:Movie {title: "Casino"})
RETURN p.name AS director_name
📦 Graph Records Returned: [{'director_name': 'Martin Scorsese'}]
💡 Final Answer: Martin Scorsese.


In [21]:
"""
Now we test multi-entity retrieval by asking who the actors in Casino were.
Notice how the LLM automatically uses the ACTED_IN relationship to return all matching performers.
"""
q2 = "Who were the actors of the movie Casino?"
res2 = qa_chain.invoke({"query": q2})

print(f"❓ Question: {q2}")
print(f"💻 Generated Cypher Query: {res2['intermediate_steps'][0]['query']}")
print(f"📦 Graph Records Returned: {res2['intermediate_steps'][1]['context']}")
print(f"💡 Final Answer: {res2['result']}")




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:ACTED_IN]->(m:Movie {title: 'Casino'})
RETURN p.name AS actor;
Full Context:
[{'actor': 'Robert De Niro'}, {'actor': 'Joe Pesci'}, {'actor': 'Sharon Stone'}, {'actor': 'James Woods'}]

> Finished chain.
❓ Question: Who were the actors of the movie Casino?
💻 Generated Cypher Query: MATCH (p:Person)-[:ACTED_IN]->(m:Movie {title: 'Casino'})
RETURN p.name AS actor;
📦 Graph Records Returned: [{'actor': 'Robert De Niro'}, {'actor': 'Joe Pesci'}, {'actor': 'Sharon Stone'}, {'actor': 'James Woods'}]
💡 Final Answer: Robert De Niro, Joe Pesci, Sharon Stone, James Woods were the actors of the movie *Casino*.


In [22]:
"""
Here we test numerical aggregation by asking how many movies Tom Hanks has acted in.
The LLM should generate an aggregation query with count() to return the exact total.
"""
q3 = "How many movies has Tom Hanks acted in?"
res3 = qa_chain.invoke({"query": q3})

print(f"❓ Question: {q3}")
print(f"💻 Generated Cypher Query: {res3['intermediate_steps'][0]['query']}")
print(f"📦 Graph Records Returned: {res3['intermediate_steps'][1]['context']}")
print(f"💡 Final Answer: {res3['result']}")




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie)
RETURN count(m) AS movieCount
Full Context:
[{'movieCount': 2}]

> Finished chain.
❓ Question: How many movies has Tom Hanks acted in?
💻 Generated Cypher Query: MATCH (p:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie)
RETURN count(m) AS movieCount
📦 Graph Records Returned: [{'movieCount': 2}]
💡 Final Answer: Tom Hanks has acted in 2 movies.


In [23]:
"""
Now we pivot to our social network data!
We ask who the friends of user John are and what posts those friends have shared.
This requires traversing from a User node across a FRIEND relationship to another User,
and then across a POSTED relationship to Post nodes.
"""
q4 = "Who are the friends of John and what posts have they made?"
res4 = qa_chain.invoke({"query": q4})

print(f"❓ Question: {q4}")
print(f"💻 Generated Cypher Query: {res4['intermediate_steps'][0]['query']}")
print(f"📦 Graph Records Returned: {res4['intermediate_steps'][1]['context']}")
print(f"💡 Final Answer: {res4['result']}")




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (john:User {name: 'John'})-[:FRIEND]->(friend:User)
OPTIONAL MATCH (friend)-[:POSTED]->(post:Post)
RETURN friend.name AS friendName, post.content AS content, post.postId AS postId, post.timestamp AS timestamp;
Full Context:
[{'friendName': 'Sarah', 'content': 'My first post!', 'postId': 2, 'timestamp': neo4j.time.DateTime(2024, 7, 14, 13, 0, 0, 0, tzinfo=<UTC>)}, {'friendName': 'Mike', 'content': 'Beautiful day in Chicago.', 'postId': 3, 'timestamp': neo4j.time.DateTime(2024, 7, 14, 13, 45, 0, 0, tzinfo=<UTC>)}, {'friendName': 'David', 'content': 'San Francisco is amazing!', 'postId': 5, 'timestamp': neo4j.time.DateTime(2024, 7, 14, 14, 30, 0, 0, tzinfo=<UTC>)}]

> Finished chain.
❓ Question: Who are the friends of John and what posts have they made?
💻 Generated Cypher Query: MATCH (john:User {name: 'John'})-[:FRIEND]->(friend:User)
OPTIONAL MATCH (friend)-[:POSTED]->(post:Post)
RETURN friend.name AS friendName, post.

In [26]:
"""
Finally, we test an analytical graph question: asking which user in our network has the highest number of friends.
The model should group users by name and count their incoming or outgoing FRIEND relationships,
ordering the results in descending order.
"""
q5 = "Which user has the most friends in the social network?"
res5 = qa_chain.invoke({"query": q5})

print(f"❓ Question: {q5}")
print(f"💻 Generated Cypher Query: {res5['intermediate_steps'][0]['query']}")
print(f"📦 Graph Records Returned: {res5['intermediate_steps'][1]['context']}")
print(f"💡 Final Answer: {res5['result']}")




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (u:User)-[:FRIEND]-(f:User)
WITH u, count(DISTINCT f) AS friendCount
RETURN u.name AS name, u.userId AS userId, friendCount
ORDER BY friendCount DESC
LIMIT 1
Full Context:
[{'name': 'Sarah', 'userId': 2, 'friendCount': 4}]

> Finished chain.
❓ Question: Which user has the most friends in the social network?
💻 Generated Cypher Query: MATCH (u:User)-[:FRIEND]-(f:User)
WITH u, count(DISTINCT f) AS friendCount
RETURN u.name AS name, u.userId AS userId, friendCount
ORDER BY friendCount DESC
LIMIT 1
📦 Graph Records Returned: [{'name': 'Sarah', 'userId': 2, 'friendCount': 4}]
💡 Final Answer: Sarah has the most friends in the social network.


In [25]:
"""
Here we compile all 5 question-answering experiments into a clean Pandas scorecard.
This gives us a high-level overview of each question asked, the generated Cypher query,
the data records retrieved, and the final synthesized response.
"""
experiments_summary = [
    {
        "ID": 1,
        "Domain": "Movies Graph",
        "Question": q1,
        "Generated Cypher": res1['intermediate_steps'][0]['query'].strip(),
        "Records Found": len(res1['intermediate_steps'][1]['context']),
        "Answer": res1['result']
    },
    {
        "ID": 2,
        "Domain": "Movies Graph",
        "Question": q2,
        "Generated Cypher": res2['intermediate_steps'][0]['query'].strip(),
        "Records Found": len(res2['intermediate_steps'][1]['context']),
        "Answer": res2['result']
    },
    {
        "ID": 3,
        "Domain": "Movies Graph",
        "Question": q3,
        "Generated Cypher": res3['intermediate_steps'][0]['query'].strip(),
        "Records Found": len(res3['intermediate_steps'][1]['context']),
        "Answer": res3['result']
    },
    {
        "ID": 4,
        "Domain": "Social Graph",
        "Question": q4,
        "Generated Cypher": res4['intermediate_steps'][0]['query'].strip(),
        "Records Found": len(res4['intermediate_steps'][1]['context']),
        "Answer": res4['result']
    },
    {
        "ID": 5,
        "Domain": "Social Graph",
        "Question": q5,
        "Generated Cypher": res5['intermediate_steps'][0]['query'].strip(),
        "Records Found": len(res5['intermediate_steps'][1]['context']),
        "Answer": res5['result']
    }
]

df_summary = pd.DataFrame(experiments_summary)
print("=== 🏆 GRAPHCYPHERQACHAIN EXPERIMENT SCORECARD ===")
display(df_summary)

print("\n🎉 Notebook 1 complete! GraphCypherQAChain successfully demonstrated across both Movie and Social graph datasets!")


=== 🏆 GRAPHCYPHERQACHAIN EXPERIMENT SCORECARD ===


,ID,Domain,Question,Generated Cypher,Records Found,Answer
0,1,Movies Graph,Who was the director of the movie Casino?,MATCH (p:Person)-[:DIRECTED]->(m:Movie {title:...,1,Martin Scorsese.
1,2,Movies Graph,Who were the actors of the movie Casino?,MATCH (p:Person)-[:ACTED_IN]->(m:Movie {title:...,4,"Robert De Niro, Joe Pesci, Sharon Stone, James..."
2,3,Movies Graph,How many movies has Tom Hanks acted in?,MATCH (p:Person {name: 'Tom Hanks'})-[:ACTED_I...,1,Tom Hanks has acted in 2 movies.
3,4,Social Graph,Who are the friends of John and what posts hav...,MATCH (john:User {name: 'John'})-[:FRIEND]->(f...,3,Sarah – “My first post!” \nMike – “Beautiful ...
4,5,Social Graph,Which user has the most friends in the social ...,"MATCH (u:User)-[:FRIEND]-(f:User)\nWITH u, cou...",1,Sarah has the most friends in the social network.



🎉 Notebook 1 complete! GraphCypherQAChain successfully demonstrated across both Movie and Social graph datasets!
